In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

sys.path.append("../src")
from datos import cargar
from filtro_adaptado import parecido, psd_en, serie_snr, snr_optimo
from plantillas import generador

In [ ]:
EVENTO_PSD = "GW150914"
DATOS = Path("../datos")
DURACION = 128
DURACION_FFT = 4
DURACION_SEGMENTO = 32
F_MIN = 20
F_MAX = 1024
MASA_MINIMA = 6
MASA_MAXIMA = 45
PARECIDO_VECINOS = 0.90
LINEAS = [(0.25, 0.0), (0.22, 0.25), (0.25, -0.2), (0.22, -0.2), (0.19, -0.2)]
SENALES_PRUEBA = 200
Q_MINIMO = 0.25
GIRO_MINIMO = -0.3
GIRO_MAXIMO = 0.4
PARECIDO_OBJETIVO = 0.95
VENTANA_MASA = 0.1

In [ ]:
datos_h1 = cargar("H1", EVENTO_PSD, DURACION, DATOS)
fs = int(datos_h1.sample_rate.value)
n = DURACION_SEGMENTO * fs
df = fs / n
indices = np.arange(int(F_MIN / df), int(F_MAX / df) + 1)
n_parecido = 2 * indices[-1]
psd = psd_en(np.arange(n // 2 + 1) * df, datos_h1.value, fs, DURACION_FFT)
plantilla = generador(indices * df, F_MIN)
senal = generador(indices * df, F_MIN, "IMRPhenomXAS")

In [ ]:
def siguiente_masa(masa, eta, giro):
    actual = np.array([plantilla(masa, eta, giro)])
    bajo, alto = masa, 1.5 * masa
    for _ in range(10):
        medio = np.sqrt(bajo * alto)
        if parecido(actual, plantilla(medio, eta, giro), psd, df, n_parecido, indices)[0] > PARECIDO_VECINOS:
            bajo = medio
        else:
            alto = medio
    return np.sqrt(bajo * alto)


def linea(eta, giro):
    masas = [MASA_MINIMA]
    while masas[-1] < MASA_MAXIMA:
        masas.append(siguiente_masa(masas[-1], eta, giro))
    return [(masa, eta, giro) for masa in masas]

In [ ]:
archivo_banco = DATOS / "banco.npz"
configuracion = repr((EVENTO_PSD, DURACION, DURACION_FFT, DURACION_SEGMENTO, F_MIN, F_MAX, MASA_MINIMA, MASA_MAXIMA, PARECIDO_VECINOS, LINEAS))
if archivo_banco.exists() and str(np.load(archivo_banco).get("configuracion")) == configuracion:
    banco = np.load(archivo_banco)["banco"]
else:
    banco = np.array([p for eta, giro in LINEAS for p in linea(eta, giro)])
    np.savez(archivo_banco, banco=banco, configuracion=configuracion)
print(f"{len(banco)} plantillas")

In [ ]:
for k, (eta, giro) in enumerate(LINEAS):
    masas = banco[(banco[:, 1] == eta) & (banco[:, 2] == giro), 0]
    plt.semilogx(masas, np.full(len(masas), k), "|", markersize=12)
plt.yticks(range(len(LINEAS)), [f"η {eta}, giro {giro}" for eta, giro in LINEAS])
plt.xlabel(r"Masa chirp (M$_\odot$)");

In [ ]:
def mejor_parecido(masa, eta, giro):
    cerca = banco[np.abs(banco[:, 0] / masa - 1) < VENTANA_MASA]
    plantillas = np.array([plantilla(*p) for p in cerca])
    return parecido(plantillas, senal(masa, eta, giro), psd, df, n_parecido, indices).max()


rng = np.random.default_rng(0)
masas_prueba = np.exp(rng.uniform(np.log(MASA_MINIMA), np.log(MASA_MAXIMA), SENALES_PRUEBA))
q = rng.uniform(Q_MINIMO, 1, SENALES_PRUEBA)
etas_prueba = q / (1 + q) ** 2
giros_prueba = rng.uniform(GIRO_MINIMO, GIRO_MAXIMO, SENALES_PRUEBA)
ajuste = np.array([mejor_parecido(*p) for p in zip(masas_prueba, etas_prueba, giros_prueba)])
print(f"Fitting factor: mediana {np.median(ajuste):.3f}, 10 % más bajo {np.percentile(ajuste, 10):.3f}, mínimo {ajuste.min():.3f}, ≥{PARECIDO_OBJETIVO}: {(ajuste >= PARECIDO_OBJETIVO).sum()}/{SENALES_PRUEBA}")

In [ ]:
plt.scatter(masas_prueba, ajuste, c=giros_prueba, cmap="coolwarm")
plt.colorbar(label="Giro de la señal")
plt.axhline(PARECIDO_OBJETIVO, color="k", linestyle="--")
plt.xscale("log")
plt.xlabel(r"Masa chirp de la señal (M$_\odot$)")
plt.ylabel("Fitting factor");